# The three arms

**Spends money.** Set `ARMS` and `PRUNES`, read section 2, then turn `DRY_RUN`
off.

`ARMS` and `PRUNES` are lists and every pair of them runs, so the number to
approve is the total at the bottom of section 2. Three arms against four
filters is twelve sweeps, and on GDC the paged ones are most of the cost.

`rerank` is Magneto: the model sees the retriever's top `TOP_K` in one prompt
and cannot decline, so its recall is capped at the retriever's recall@`TOP_K`.

`gated` shows the same block and allows the reply NONE. `rerank` against
`gated` is what declining is worth at fixed depth.

`paged` shows `WINDOW` candidates at a time; the model commits or asks for the
next block. `gated` against `paged` is what depth is worth. Cost per column:

| the column | blocks it opens |
|---|---|
| match in the first window | 1 |
| match deep in the tail | `ceil(rank / WINDOW)` |
| no match anywhere | `ceil(n_targets / WINDOW)`, so 37 on GDC |

404 of GDC's 569 source columns have no gold target, so the last row is the
common case.

`prompt_layout` is fixed by the one-call arms (`oneshot` for rerank,
`oneshot_none` for gated); changing it makes a one-block paged run.

## 0. What to run

In [1]:
import sys
from pathlib import Path

import pandas as pd

# Wide enough that `arm` is not collapsed into an ellipsis.
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO))

from corpora import load
from evaluation import scoring as report
from schemaoutis import cache, pruning
from schemaoutis.bench import Bench

# --- what to run ----------------------------------------------------------
CORPUS = "valentine/chembl"                  # or "valentine", "valentine/opendata", ...
STUDIES = None                  # None = every pair in the corpus
REPS = 1                        # runs per configuration; 3 for a spread

# Every arm runs against every filter in PRUNES, all priced together in
# section 2. Cheapest first, so an interrupted sweep has whole arms done.
ARMS = ["paged"]

TOP_K = 20                      # the shortlist rerank and gated are shown

# --- paged only; ignored by rerank and gated ------------------------------
WINDOW = 20                     # candidates per block
# A block scoring below this is declined; 0.00 leaves it to the model.
THRESHOLDS = [0.00]

# --- the filter -----------------------------------------------------------
#   None               every column asked, every candidate offered
#   "mutual"           skip a source column with no mutual nearest neighbour
#                      (41% of the calls on GDC)
#   "zero"             drop targets no source is close to; smaller blocks
#   ("mutual","zero")  both
# Each filter is its own configuration in the filename and gets its own Bench.
# `bench.cost` quotes the unfiltered worst case.
#PRUNES = [None, "mutual", "zero", "zerosrc", ("mutual","zero"), ("zerosrc","zero")]
PRUNES = [None]

# What a walk does with an unreadable reply: "stop" leaves the column
# unanswered; "fallback" keeps the retriever's window, as rerank does. Both
# send the same requests, so this is not in the filename.
ON_UNPARSEABLE = "stop"

# Re-ask an unreadable block up to this many times, with the format spelled
# out. Tagged into the variant as `retry{n}`; seed those files first with
# `reseed.plan` + `reseed.write`, or section 1 refuses to run.
MAX_PARSE_RETRIES = 3

# `match_retriever` shows the text the order was built from; `magneto` uses
# Magneto's own. Either way it is in the variant.
TARGET_TEXT = "match_retriever"

# True prices the sweep and sends nothing.
DRY_RUN = False

# label -> the four knobs that decide the order. Labels must be unique: a
# cached order refuses to be served under a different configuration.
RETRIEVERS = {
    "qwen": dict(embedding_model="Qwen/Qwen3-Embedding-4B",
                 encoding_mode="header_values_xml",
                 sampling_mode="mixed", sampling_size=10, seed=0),
    #"magneto": dict(embedding_model="vida-nyu/magneto-schema-retriever-gdc",
    #                 encoding_mode="header_values_default",
    #                 sampling_mode="mixed", sampling_size=10, seed=0),
}

# `client` goes to build_client; on Ollama, `think=False` and `num_ctx` both
# change the answer. A `jev-` model uses the decisions endpoint instead: pin
# the version, keep MAX_PARSE_RETRIES at 0 (a typed answer cannot fail to
# parse) and run paged at threshold 0.00 only, since its scores are one
# distribution per block.
MODELS = [
    dict(provider="anthropic", model="claude-sonnet-5",
         prompt_layout="standard"),
    #dict(provider="openrouter", model="jev-1.13", prompt_layout="standard"),
    #dict(provider="ollama", model="deepseek-v4-flash:cloud",
    #     prompt_layout="standard", client=dict(think=False, timeout_s=600.0)),
    #dict(provider="ollama", model="deepseek-v4.1-flash:cloud",
    #     prompt_layout="standard", client=dict(think=False, timeout_s=600.0)),
    #dict(provider="ollama", model="deepseek-v4-pro:cloud",
    #     prompt_layout="standard", client=dict(think=False, timeout_s=600.0)),
    #dict(provider="openrouter", model="anthropic/claude-sonnet-5",
    #     record_as="anthropic", prompt_layout="standard"),
]

def levels_for(arm):
    """The knob this arm sweeps. Only `paged` has a threshold to vary."""
    return THRESHOLDS if arm == "paged" else [TOP_K]


PAIRS = load(CORPUS, names=STUDIES)

# filter tag -> the Bench that writes under it.
BENCHES = {pruning.tag(prune): Bench(
    corpus=CORPUS, pairs=PAIRS,
    results=REPO / "results" / "runs",
    retrievers=RETRIEVERS, models=MODELS, target_text=TARGET_TEXT,
    reps=REPS, window=WINDOW, seed=0, on_unparseable=ON_UNPARSEABLE,
    prune=prune, max_parse_retries=MAX_PARSE_RETRIES,
    dry_run=DRY_RUN,
    resume=True, use_cache=True, verbose=True,
) for prune in PRUNES}

# (arm, filter tag, bench), arm by arm.
JOBS = [(arm, tag, BENCHES[tag]) for arm in ARMS for tag in BENCHES]

# For helpers that read only a frame, such as `health`.
BENCH = next(iter(BENCHES.values()))


def label(arm, tag):
    return f"{arm}, prune={tag or 'none'}"


BENCH.banner()
print(f"\narms       {ARMS}")
print(f"filters    {[t or 'none' for t in BENCHES]}")
print(f"\n{len(JOBS)} job(s), all of them priced together in section 2:")
for arm, tag, _ in JOBS:
    print(f"  {label(arm, tag):<28} levels={levels_for(arm)}")

env        ['ANTHROPIC_API_KEY', 'OPENROUTER_API_KEY']
corpus     valentine/chembl: 180 studies, 3060 source columns
retrievers ['qwen']
models     ['anthropic__claude-sonnet-5__tgt-match-retriever']
target     match_retriever
prune      none
retries    3 on an unreadable reply   (tagged into the variant)
repeats    1   runs -> C:\Users\muril\SchemaOutis\results\runs

DRY_RUN = False   *** THIS WILL SEND REQUESTS ***

arms       ['paged']
filters    ['none']

1 job(s), all of them priced together in section 2:
  paged, prune=none            levels=[0.0]


## 1. What this will send

`bench.cost` reads the runs on disk and quotes only the columns that are
missing or that a retry could change. Nothing is sent until section 2.


In [2]:
# `levels` is not optional: without it every level on disk is counted, and
# a sweep at w=20, tau=0 was quoted columns belonging to a w=5, tau=0.5 run.
TODO = pd.concat([bench.cost(arm, levels=levels_for(arm)).assign(
    arm=arm, prune=tag or "none") for arm, tag, bench in JOBS],
    ignore_index=True)
display(TODO)
print("")
print(f"this configuration would send "
      f"{int(TODO['columns_todo'].sum()):,} column(s)")


0 run(s) to do
  columns            5
  best case          5 LLM calls
  worst case        10 LLM calls


,retriever,model,threshold,runs_todo,runs_partial,columns_todo,columns_stuck,calls_best,calls_worst,arm,prune
0,qwen,anthropic__claude-sonnet-5__tgt-match-retriever,0.0,0,0,5,0,5,10,paged,none



this configuration would send 5 column(s)


## 3. Run

In [3]:
# `run` puts the arm and the filter on every row, so the sweeps concatenate
# without anything being added here and no row loses which job made it.
SWEEPS = []
for arm, tag, bench in JOBS:
    print(f"\n=== {label(arm, tag)} " + "=" * 40)
    SWEEPS.append(bench.run(arm, levels=levels_for(arm)))

SWEEP = pd.concat(SWEEPS, ignore_index=True) if SWEEPS else pd.DataFrame()
display(SWEEP)


=== paged, prune=none ========================================
[qwen 1/180] ChEMBL/Joinable/assays_both_50_1_ac1_ev: 12 x 12 targets (0.0s)
[qwen 2/180] ChEMBL/Joinable/assays_both_50_1_ac2_ev: 12 x 12 targets (0.0s)
[qwen 3/180] ChEMBL/Joinable/assays_both_50_1_ac3_ev: 12 x 12 targets (0.0s)
[qwen 4/180] ChEMBL/Joinable/assays_both_50_1_ac4_ev: 12 x 12 targets (0.0s)
[qwen 5/180] ChEMBL/Joinable/assays_both_50_1_ac5_ev: 12 x 12 targets (0.0s)
[qwen 6/180] ChEMBL/Joinable/assays_both_50_1_ec_ev: 12 x 12 targets (0.0s)
[qwen 7/180] ChEMBL/Joinable/assays_both_50_30_ac1_ev: 14 x 15 targets (0.0s)
[qwen 8/180] ChEMBL/Joinable/assays_both_50_30_ac2_ev: 14 x 15 targets (0.0s)
[qwen 9/180] ChEMBL/Joinable/assays_both_50_30_ac3_ev: 14 x 15 targets (0.0s)
[qwen 10/180] ChEMBL/Joinable/assays_both_50_30_ac4_ev: 14 x 15 targets (0.0s)
[qwen 11/180] ChEMBL/Joinable/assays_both_50_30_ac5_ev: 14 x 15 targets (0.0s)
[qwen 12/180] ChEMBL/Joinable/assays_both_50_30_ec_ev: 14 x 15 targets (0.0s)
[qwen

,pair,arm,cached,columns,committed,accepted,abstained,errored,complete,calls,tokens_in,tokens_out,wall_s,retriever,model,rep,prune,threshold
0,ChEMBL/Joinable/assays_both_50_1_ac1_ev,paged,True,12,2,2,10,0,True,12,16657,872,0.0,qwen,anthropic__claude-sonnet-5__tgt-match-retriever,1,,0.0
1,ChEMBL/Joinable/assays_both_50_1_ac2_ev,paged,True,12,5,5,7,0,True,12,16201,978,0.0,qwen,anthropic__claude-sonnet-5__tgt-match-retriever,1,,0.0
2,ChEMBL/Joinable/assays_both_50_1_ac3_ev,paged,True,12,3,3,9,0,True,12,16489,578,0.0,qwen,anthropic__claude-sonnet-5__tgt-match-retriever,1,,0.0
3,ChEMBL/Joinable/assays_both_50_1_ac4_ev,paged,True,12,4,4,8,0,True,12,16477,1999,0.0,qwen,anthropic__claude-sonnet-5__tgt-match-retriever,1,,0.0
4,ChEMBL/Joinable/assays_both_50_1_ac5_ev,paged,True,12,3,3,9,0,True,12,16777,5040,0.0,qwen,anthropic__claude-sonnet-5__tgt-match-retriever,1,,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
175,ChEMBL/View-Unionable/assays_both_0_70_ac4_ev,paged,True,19,17,17,2,0,True,19,34189,1277,0.0,qwen,anthropic__claude-sonnet-5__tgt-match-retriever,1,,0.0
176,ChEMBL/View-Unionable/assays_both_0_70_ac5_av,paged,False,19,17,17,2,0,True,19,40329,2112,2.2,qwen,anthropic__claude-sonnet-5__tgt-match-retriever,1,,0.0
177,ChEMBL/View-Unionable/assays_both_0_70_ac5_ev,paged,True,19,17,17,2,0,True,19,34221,2474,0.0,qwen,anthropic__claude-sonnet-5__tgt-match-retriever,1,,0.0
178,ChEMBL/View-Unionable/assays_both_0_70_ec_av,paged,True,19,18,18,1,0,True,19,39035,1201,0.0,qwen,anthropic__claude-sonnet-5__tgt-match-retriever,1,,0.0


## 4. Did it work

In [4]:
# Groups by arm and by filter when either varies, so two jobs are never
# pooled into one row.
display(BENCH.health(SWEEP))

no errored columns

4 run(s) have unanswered columns:
                                          model                                                  pair  columns  committed  abstained  errored
anthropic__claude-sonnet-5__tgt-match-retriever ChEMBL/Semantically-Joinable/assays_both_50_30_ac5_av       14         10          3        0
anthropic__claude-sonnet-5__tgt-match-retriever ChEMBL/Semantically-Joinable/assays_both_50_50_ac2_av       17         12          4        0
anthropic__claude-sonnet-5__tgt-match-retriever  ChEMBL/Semantically-Joinable/assays_both_50_50_ec_av       17         14          2        0
anthropic__claude-sonnet-5__tgt-match-retriever ChEMBL/Semantically-Joinable/assays_vertical_50_ec_av       17         13          3        0


,,,runs,columns,calls,committed,abstained,wall_s,calls_per_column
retriever,model,threshold,,,,,,,
qwen,anthropic__claude-sonnet-5__tgt-match-retriever,0.0,180,3060,3076,2403,653,20.1,1.01


## 5. Matching quality, and what declining bought

In [5]:
# `score` takes an inventory narrowed to one arm and is told which, so this
# is a loop and not one call over the concatenation. `kind` is the arm; the
# inventory's own `arm` column is the model tag, which is a different thing.
INVS, SCORES = [], []
for arm, tag, bench in JOBS:
    inv = bench.inventory(arm)          # already narrowed to this filter
    if not len(inv):
        print(f"{label(arm, tag)}: nothing on disk")
        continue
    INVS.append(inv)
    SCORES.append(report.score(inv, bench.pairs, results=bench.results,
                               corpus=CORPUS, kind=arm)
                  .assign(kind=arm, prune=tag))

INV = pd.concat(INVS, ignore_index=True) if INVS else pd.DataFrame()
SCORED = pd.concat(SCORES, ignore_index=True) if SCORES else pd.DataFrame()

# Both level columns, because the arms do not share one: `paged` sweeps
# `threshold` and the one-call arms only have `top_k`. Every list is filtered
# by what is actually present, so a single-arm sweep prints one of them.
IDS = ["kind", "prune", "retriever", "model", "top_k", "threshold", "rep"]
QUALITY = IDS + ["n_studies", "n_columns", "n_gold_pairs", "mrr", "recall@1",
                 "recall@20", "top1", "ret_mrr", "ret_recall@20"]
DECIDING = IDS + ["precision_committed", "precision_evaluated", "f1_committed",
                  "n_declined", "n_pruned", "n_model_declined",
                  "n_without_gold", "abstain_precision", "abstain_recall"]
COSTING = IDS + ["calls", "calls_per_column", "calls_median", "calls_max",
                 "calls_share_top", "calls_with_gold", "calls_without_gold",
                 "n_fallback"]

print("ranking quality")
display(SCORED[[c for c in QUALITY if c in SCORED.columns]].round(3))
print("\nwhat declining buys")
display(SCORED[[c for c in DECIDING if c in SCORED.columns]].round(3))
print("\nwhat it costs")
display(SCORED[[c for c in COSTING if c in SCORED.columns]].round(3))

ranking quality


,kind,prune,retriever,model,threshold,rep,n_studies,n_columns,n_gold_pairs,mrr,recall@1,recall@20,top1,ret_mrr,ret_recall@20
0,paged,,magneto,openrouter__jev-1.13__tgt-match-retriever,0.0,1,180,3060,2052,0.867,0.862,0.873,0.862,0.691,0.995
1,paged,,qwen,anthropic__claude-sonnet-5__tgt-match-retriever,0.0,1,180,3060,2052,0.951,0.942,0.966,0.942,0.921,1.000
2,paged,,qwen,ollama__deepseek-v4.1-flash-cloud__tgt-match-r...,0.0,1,180,3060,2052,0.935,0.914,0.973,0.914,0.921,1.000
3,paged,,qwen,openrouter__jev-1.13__tgt-match-retriever,0.0,1,180,3060,2052,0.894,0.885,0.903,0.885,0.921,1.000



what declining buys


,kind,prune,retriever,model,threshold,rep,precision_committed,precision_evaluated,f1_committed,n_declined,n_pruned,n_model_declined,n_without_gold,abstain_precision,abstain_recall
0,paged,,magneto,openrouter__jev-1.13__tgt-match-retriever,0.0,1,0.796,0.595,0.828,837,0,837,1008,0.742,0.616
1,paged,,qwen,anthropic__claude-sonnet-5__tgt-match-retriever,0.0,1,0.804,0.640,0.867,653,0,653,1008,0.949,0.615
2,paged,,qwen,ollama__deepseek-v4.1-flash-cloud__tgt-match-r...,0.0,1,0.706,0.624,0.797,406,0,406,1008,0.869,0.350
3,paged,,qwen,openrouter__jev-1.13__tgt-match-retriever,0.0,1,0.757,0.610,0.816,660,0,660,1008,0.838,0.549



what it costs


,kind,prune,retriever,model,threshold,rep,calls,calls_per_column,calls_median,calls_max,calls_share_top,calls_with_gold,calls_without_gold,n_fallback
0,paged,,magneto,openrouter__jev-1.13__tgt-match-retriever,0.0,1,3156,1.03,1.0,2,0.127,1.047,1.0,0
1,paged,,qwen,anthropic__claude-sonnet-5__tgt-match-retriever,0.0,1,3076,1.01,1.0,2,0.105,1.008,1.0,0
2,paged,,qwen,ollama__deepseek-v4.1-flash-cloud__tgt-match-r...,0.0,1,3092,1.01,1.0,2,0.109,1.016,1.0,0
3,paged,,qwen,openrouter__jev-1.13__tgt-match-retriever,0.0,1,3117,1.02,1.0,2,0.116,1.028,1.0,0


## 6. As a blocking scheme

Only `paged` and `gated` make a block decision, so `rerank` rows are dropped
here rather than shown as zeros.

Pair Completeness is reported twice: `pc_traversed` is the gold the walk was
shown and `pc_accepted` the gold that survived the decision. The gap between
them is recall paid for and then thrown away. Reduction Ratio counts
comparisons saved and cannot see which ones happened, so it rises for free with
every column a filter removes and is not interpretable without precision beside
it.

In [6]:
BLOCKING = IDS + ["pc_traversed", "pc_accepted", "rr", "rr_stop_at_gold"]
CLASSIFIER = IDS + ["block_precision_gold", "block_recall_gold",
                    "tp_gold", "fp_gold", "fn_gold",
                    "n_comparisons", "n_comparisons_brute"]

# Only the arms that make a block decision, rather than the whole frame: a
# rerank row carries these as NaN and printing it would read as a zero.
BLOCKS = (SCORED[SCORED["pc_traversed"].notna()]
          if len(SCORED) and "pc_traversed" in SCORED.columns
          else SCORED.iloc[:0] if len(SCORED) else SCORED)

if len(BLOCKS):
    print("the survey's measures")
    display(BLOCKS[[c for c in BLOCKING if c in BLOCKS.columns]].round(3))
    print("\nthe model as a block classifier, over answerable columns only")
    display(BLOCKS[[c for c in CLASSIFIER if c in BLOCKS.columns]].round(3))
    print("\ngold seen and then discarded, per configuration:")
    lost = (BLOCKS["pc_traversed"] - BLOCKS["pc_accepted"]).round(3)
    for r, l in zip(BLOCKS.to_dict("records"), lost):
        level = r.get("threshold", r.get("top_k"))
        print(f"  {r['kind']:<7} p{r['prune'] or 'none':<12} "
              f"{report.short_model(r['model']):<44} {level}  {l:.3f}")
else:
    print("no arm here makes a block decision, so there is nothing to show.")

the survey's measures


,kind,prune,retriever,model,threshold,rep,pc_traversed,pc_accepted,rr,rr_stop_at_gold
0,paged,,magneto,openrouter__jev-1.13__tgt-match-retriever,0.0,1,1.0,0.891,0.039,0.044
1,paged,,qwen,anthropic__claude-sonnet-5__tgt-match-retriever,0.0,1,1.0,0.981,0.044,0.044
2,paged,,qwen,ollama__deepseek-v4.1-flash-cloud__tgt-match-r...,0.0,1,1.0,0.974,0.043,0.044
3,paged,,qwen,openrouter__jev-1.13__tgt-match-retriever,0.0,1,1.0,0.945,0.041,0.044



the model as a block classifier, over answerable columns only


,kind,prune,retriever,model,threshold,rep,block_precision_gold,block_recall_gold,tp_gold,fp_gold,fn_gold,n_comparisons,n_comparisons_brute
0,paged,,magneto,openrouter__jev-1.13__tgt-match-retriever,0.0,1,0.996,0.891,1829,7,223,53676,55872
1,paged,,qwen,anthropic__claude-sonnet-5__tgt-match-retriever,0.0,1,0.999,0.981,2013,2,39,53436,55872
2,paged,,qwen,ollama__deepseek-v4.1-flash-cloud__tgt-match-r...,0.0,1,1.000,0.974,1999,0,53,53484,55872
3,paged,,qwen,openrouter__jev-1.13__tgt-match-retriever,0.0,1,0.997,0.945,1939,6,113,53559,55872



gold seen and then discarded, per configuration:
  paged   pnone         jev-1.13 tgt-match-retriever                 0.0  0.109
  paged   pnone         claude-sonnet-5 tgt-match-retriever          0.0  0.019
  paged   pnone         deepseek-v4.1-flash-cloud tgt-match-retriever 0.0  0.026
  paged   pnone         jev-1.13 tgt-match-retriever                 0.0  0.055


## 7. Where the calls went

The cost story is a distribution, not a mean. A column that accepts its first
block costs one call and a column with no match anywhere costs
`ceil(n_targets / window)`, so the average sits between two numbers that almost
nothing actually pays.

In [7]:
# Grouped by the job too. Without `kind` a rerank run at one call per column
# would be averaged into a paged run at nineteen, and the mean would describe
# neither.
KEYS = ["retriever", "provider", "arm", "kind", "prune"]
COLUMNS = pd.concat(
    [report.per_column(runs.itertuples(), BENCH.pairs).assign(
        retriever=k[0], model=cache.arm_tag(k[1], k[2]), kind=k[3], prune=k[4])
     for k, runs in INV.groupby(KEYS)],
    ignore_index=True) if len(INV) else pd.DataFrame()

if len(COLUMNS):
    display(COLUMNS.groupby(["kind", "prune", "model"])["api_calls"]
            .describe().round(2))
    print("\ncalls by whether the column had a correct target at all:")
    display(COLUMNS.groupby(["kind", "prune", "model", "has_gold"])["api_calls"]
            .agg(["count", "mean", "median", "max"]).round(2))

count  mean   std  min  25%  50%  75%  max
kind  prune model                                                                                          
paged       anthropic__claude-sonnet-5__tgt-match-retriever     3060.0  1.01  0.07  1.0  1.0  1.0  1.0  2.0
            ollama__deepseek-v4.1-flash-cloud__tgt-match-re...  3060.0  1.01  0.10  1.0  1.0  1.0  1.0  2.0
            openrouter__jev-1.13__tgt-match-retriever           6120.0  1.02  0.16  1.0  1.0  1.0  1.0  2.0


calls by whether the column had a correct target at all:


count  mean  median  max
kind  prune model                                              has_gold                          
paged       anthropic__claude-sonnet-5__tgt-match-retriever    False      1008  1.00     1.0    1
                                                               True       2052  1.01     1.0    2
            ollama__deepseek-v4.1-flash-cloud__tgt-match-re... False      1008  1.00     1.0    1
                                                               True       2052  1.02     1.0    2
            openrouter__jev-1.13__tgt-match-retriever          False      2016  1.00     1.0    1
                                                               True       4104  1.04     1.0    2

## 8. Spread across repeats

A gap between two models smaller than either one's own spread is not a gap, and
one run cannot say which case it is in.

In [8]:
if len(SCORED) and SCORED["rep"].nunique() > 1:
    display(report.across_repeats(SCORED).round(4))
else:
    print(f"{SCORED['rep'].nunique() if len(SCORED) else 0} repeat(s) on disk; "
          "set REPS above 1 to measure the noise floor.")

1 repeat(s) on disk; set REPS above 1 to measure the noise floor.
